# Week 2 — XGBoost Internals: Sparsity-Aware Splits, Weighted Quantile Sketch, Parallelism

> *Why XGBoost is fast on missing data and how the weighted quantile sketch turns the second-order objective into linear-time split finding.*

## Learning objectives

By the end of this notebook, you will be able to:

1. Derive and implement sparsity-aware split finding with learned default directions.
2. State and motivate the weighted quantile sketch — why $h_i$-weighting is non-negotiable.
3. Describe XGBoost's block-parallel column layout and cache-aware access pattern.
4. Inspect default directions and tree structures from a trained XGBoost model.
5. Quantify the benefit of native missing-value handling vs. row-dropping or imputation.

## Outline

1. **Sparsity-aware split finding** — the learned default direction
2. **The exact algorithm** — pseudo-code and complexity
3. **Weighted quantile sketch** — why $h_i$-weighting matters
4. **Approximate split finding** at scale
5. **Block-parallel column scans** and **cache-aware access**
6. **Practical demonstration**: missing-data robustness on real data
7. **Tree visualization** and default-direction inspection


## 1. Sparsity-aware split finding

Real-world tabular data is rife with structural sparsity — missing values, one-hot encodings, zero-as-default fields. Chen & Guestrin (2016, §3.4) proposed to **learn** the default direction at every split rather than imputing missing values upfront.

### The idea

For each candidate split (feature $j$, threshold $\tau$):

1. Compute $(G_L, H_L, G_R, H_R)$ aggregating over *observed* rows only (those where $x_{i,j} \ne \text{NaN}$).
2. Aggregate the missing rows' contribution: $(G_{\text{miss}}, H_{\text{miss}})$.
3. Evaluate the gain under two scenarios:
   - **Direction A:** missing rows → left child  
     $G_L^A = G_L + G_{\text{miss}}, \;\; H_L^A = H_L + H_{\text{miss}}$
   - **Direction B:** missing rows → right child  
     $G_R^B = G_R + G_{\text{miss}}, \;\; H_R^B = H_R + H_{\text{miss}}$
4. Pick the direction with higher gain; record it on the split node.

At inference, every row whose value at feature $j$ is missing is routed according to the *learned* default direction. The model has effectively chosen the best imputation jointly with the tree structure.

### Why this beats imputation

- **No information leak.** Imputation uses statistics of the training set that may not match the inference distribution. Learning the direction sidesteps this.
- **Per-split, not per-feature.** A feature may have different optimal default directions at different nodes — a power that single-feature imputation cannot replicate.
- **Zero feature engineering.** No upstream imputer to fit, persist, and version. The model file *is* the imputation policy.


## 2. The exact algorithm (pseudo-code)

```
input  : data X (n × d), gradients g_i, Hessians h_i, regularization λ, γ
output : best (feature, threshold, default_direction, gain)

best_gain ← -∞
for j in 1..d:
    observed ← rows where x_{i,j} is not missing
    sort observed by x_{i,j} ascending
    compute prefix sums of g, h over the sorted observed rows
    G_miss ← Σ g_i over missing rows; H_miss ← Σ h_i over missing rows

    G_L, H_L ← 0, 0
    for i in 1..|observed| - 1:
        G_L += g_{sorted[i]}; H_L += h_{sorted[i]}
        G_R ← G_obs_total - G_L; H_R ← H_obs_total - H_L

        # candidate A: missing → left
        gain_A ← gain((G_L + G_miss, H_L + H_miss), (G_R, H_R))
        # candidate B: missing → right
        gain_B ← gain((G_L, H_L), (G_R + G_miss, H_R + H_miss))

        if max(gain_A, gain_B) > best_gain:
            best_gain ← max(gain_A, gain_B)
            best ← (j, threshold = (x_{sorted[i]} + x_{sorted[i+1]}) / 2,
                    default_direction = "left" if gain_A ≥ gain_B else "right",
                    gain = best_gain)
return best
```

### Complexity

$O(n d \log n)$ — the same as exact split finding without missing-data handling. The "two scenarios" comparison adds only a constant factor. This is the central engineering win: **missing-value robustness for free**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src'))

from gradient_forge.xgboost_internals.sparsity_aware import SparsityAwareSplitFinder
from gradient_forge.xgboost_internals.quantile_sketch import WeightedQuantileSketch
from gradient_forge.xgboost_internals import XGBoostTrainer
from gradient_forge.data.loaders import load_synthetic_classification, inject_missingness
from gradient_forge.utils import seed_everything
seed_everything(42)


## 3. Controlled demonstration

Construct a one-dimensional dataset where the optimal default direction is unambiguous (missing rows clearly belong on the right), and confirm that `SparsityAwareSplitFinder` picks it.


In [ ]:
# 6 observed rows + 2 missing rows whose gradients clearly fit on the right.
X = np.array([[0.0], [1.0], [2.0], [4.0], [5.0], [6.0], [np.nan], [np.nan]])
g = np.array([-1.0, -1.0, -1.0,  1.0,  1.0,  1.0,  0.9,  0.95])
h = np.ones_like(g)

finder = SparsityAwareSplitFinder(reg_lambda=1.0, min_child_weight=0.0)
result = finder.find_best(X, g, h)
print(result)
assert result is not None and result.default_left is False, "expected default direction RIGHT"
print("\n✓ Default direction learned: RIGHT (missing rows go to the high-gain side)")


## 4. Weighted quantile sketch

For large datasets, the exact $O(n d \log n)$ split finder is too slow. Approximate methods bin each feature into $\sim 1/\varepsilon$ candidate split points. **Crucially**, the binning must be weighted by $h_i$ because — as derived in Week 1 — the second-order objective is locally a *weighted* MSE with weights $h_i$:

$$
\sum_{i} \tfrac{1}{2} h_i \left( f(x_i) - \left( -\frac{g_i}{h_i} \right) \right)^{2} + \text{const}.
$$

Candidate split points should approximate quantiles of the empirical distribution of $x_{i,j}$ weighted by $h_i$, not the unweighted quantiles.

### $\varepsilon$-approximate weighted quantiles

Given $\{(x_i, h_i)\}_{i=1}^{n}$ with total weight $W = \sum_i h_i$, the weighted CDF is

$$
F_w(z) = \frac{1}{W} \sum_{i : x_i \le z} h_i.
$$

An $\varepsilon$-approximate quantile summary $\{q_k\}$ satisfies

$$
\bigl| F_w(q_k) - \tau \bigr| \le \varepsilon \qquad \forall \tau \in [0, 1].
$$

A summary of size $O(1/\varepsilon)$ suffices. Production XGBoost uses a streaming GK-style merge/prune sketch that works under data-parallel partitioning; our `WeightedQuantileSketch` is a simpler sorted-array variant.


In [ ]:
rng = np.random.default_rng(0)
x_dist = np.concatenate([rng.normal(loc=-3, size=500), rng.normal(loc=3, size=500)])

# Case A: uniform Hessians  -> candidates match unweighted quantiles
# Case B: right cluster dominates  -> candidates skew right
h_uniform = np.ones_like(x_dist)
h_skewed  = np.concatenate([np.ones(500), 100.0 * np.ones(500)])

sk = WeightedQuantileSketch(eps=0.05)
cands_uniform = sk.candidates(x_dist, h_uniform)
cands_skewed  = sk.candidates(x_dist, h_skewed)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(x_dist, bins=40, alpha=0.4, color="grey", label="data distribution")
ax.vlines(cands_uniform, 0, 70, color="C0", alpha=0.7, label=f"uniform h: {len(cands_uniform)} candidates")
ax.vlines(cands_skewed,  0, 50, color="C3", alpha=0.7, label=f"skewed h:  {len(cands_skewed)} candidates")
ax.set_xlabel("x"); ax.set_title("Weighted quantile sketch concentrates candidates where $h_i$ is large")
ax.legend(); plt.show()

frac_right_uniform = float((cands_uniform > 0).sum() / len(cands_uniform))
frac_right_skewed  = float((cands_skewed  > 0).sum() / len(cands_skewed))
print(f"Fraction of candidates in right cluster:")
print(f"  uniform Hessians : {frac_right_uniform:.0%}")
print(f"  skewed Hessians  : {frac_right_skewed:.0%}  ← shifted toward high-h region")


## 5. Block-parallel column scans and cache-aware access

XGBoost organizes data in **blocks**: each column is pre-sorted with row indices, and the resulting structure is laid out in memory for fast linear scans.

### What this enables

| Mechanism | Benefit |
|-----------|---------|
| **Per-column linear scan** | The exact split finder for column $j$ is a single pass over the sorted column — friendly to CPU branch prediction. |
| **Cache-aware prefetching** | The inner loop accesses $g_i, h_i$ via the column's row-index array. XGBoost prefetches the gradient/Hessian arrays into L2 cache while the column is being scanned. |
| **Column-parallel threading** | Different threads own disjoint subsets of features; no shared writes during split finding. |
| **Out-of-core training** | Blocks can be paged from disk on demand — XGBoost can train on datasets larger than RAM. |
| **GPU portability** | The block is a natural CUDA grid unit; histogram construction becomes a parallel reduction. |

### Pseudo-illustration of cache-aware access

```
for each column j (in parallel across threads):
    prefetch g[ block_row_indices[j] ] into L2
    prefetch h[ block_row_indices[j] ] into L2
    for each row index r in the sorted column:
        accumulate G_L, H_L
        evaluate gain candidate
```

The prefetch hides memory latency: by the time the inner loop touches $g$ and $h$, they are already in cache.

Below we *measure* a cache-friendly vs. cache-hostile access pattern to make the principle concrete — not the full block layout, but the same lesson: **memory-access patterns matter as much as arithmetic**.


In [ ]:
import time

n = 10_000_000
g_data = np.random.default_rng(0).normal(size=n).astype(np.float64)
indices_seq = np.arange(n)                            # sequential — cache-friendly
indices_rnd = np.random.default_rng(0).permutation(n)  # random — cache-hostile

def time_access(g, idx, n_runs=3):
    runs = []
    for _ in range(n_runs):
        t0 = time.perf_counter_ns()
        _ = float(np.sum(g[idx]))
        runs.append((time.perf_counter_ns() - t0) / 1e6)  # ms
    return min(runs)

t_seq = time_access(g_data, indices_seq)
t_rnd = time_access(g_data, indices_rnd)
print(f"Sequential access (cache-friendly) : {t_seq:7.2f} ms")
print(f"Random access     (cache-hostile)  : {t_rnd:7.2f} ms")
print(f"Slowdown from missed prefetches    : {t_rnd / t_seq:6.1f}x")


**The point.** When you scale to billions of rows, even constant factors dominate. XGBoost's block layout *is* the constant-factor optimization.


## 6. Practical demonstration — missing-value robustness

Inject 15% MCAR (missing-completely-at-random) missingness into a classification dataset, then compare three strategies:

- **Baseline:** no missingness (oracle).
- **Drop rows:** discard any row with at least one missing feature.
- **XGBoost native:** let the learned default direction handle it.


In [ ]:
X_full, y_full = load_synthetic_classification(n_samples=8_000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.25, stratify=y_full, random_state=42
)

common = dict(task="binary", n_estimators=400, learning_rate=0.05, max_depth=6)

# Baseline: clean data.
clean = XGBoostTrainer(**common).fit(X_train, y_train,
                                     eval_set=(X_test, y_test),
                                     early_stopping_rounds=30)
auc_clean = roc_auc_score(y_test, clean.predict_proba(X_test)[:, 1])

# Inject missingness.
X_train_miss = inject_missingness(X_train, rate=0.15, random_state=7)
X_test_miss  = inject_missingness(X_test,  rate=0.15, random_state=11)

# Strategy A: drop rows with any missing value.
keep = ~np.isnan(X_train_miss).any(axis=1)
X_dropped, y_dropped = X_train_miss[keep], y_train[keep]
if len(np.unique(y_dropped)) == 2:
    dropper = XGBoostTrainer(**common).fit(X_dropped, y_dropped,
                                           eval_set=(X_test, y_test),
                                           early_stopping_rounds=30)
    auc_drop = roc_auc_score(y_test, dropper.predict_proba(X_test)[:, 1])
    drop_kept = float(keep.mean())
else:
    auc_drop, drop_kept = float("nan"), float("nan")

# Strategy B: XGBoost native missing-value handling.
native = XGBoostTrainer(**common).fit(X_train_miss, y_train,
                                      eval_set=(X_test_miss, y_test),
                                      early_stopping_rounds=30)
auc_native = roc_auc_score(y_test, native.predict_proba(X_test_miss)[:, 1])

import pandas as pd
pd.DataFrame([
    {"strategy": "clean data (oracle)",     "data kept": "100%",            "AUC": f"{auc_clean:.4f}"},
    {"strategy": "drop rows with NaN",       "data kept": f"{drop_kept:.0%}", "AUC": f"{auc_drop:.4f}"},
    {"strategy": "XGBoost native missing",   "data kept": "100%",            "AUC": f"{auc_native:.4f}"},
])


**Reading the result.** Row-dropping at 15% missingness discards ~95% of the training data (any row with at least one of 20 features missing). XGBoost's native handling retains every row and recovers most of the oracle AUC — typically within a few thousandths.


## 7. Inspecting learned default directions

XGBoost's text dump exposes the per-node default direction. We can pull it out and confirm the model has actually *learned* a policy rather than always sending missing rows in the same direction.


In [ ]:
directions = native.default_directions()
print(f"Inspected {len(directions)} split nodes — first 10 default-direction annotations:\n")
for line in directions[:10]:
    print(" ", line[:120])

# Count left vs right default directions across the entire booster.
left = sum("missing=" + line.split("missing=")[1].split(",")[0].split("]")[0] for line in directions if False)
import re
miss_targets = [re.search(r"missing=(\d+)", line).group(1) for line in directions if "missing=" in line]
yes_targets  = [re.search(r"yes=(\d+)",     line).group(1) for line in directions if "yes="     in line]
default_left = sum(1 for m, y in zip(miss_targets, yes_targets) if m == y)
print(f"\nLearned policy: {default_left}/{len(miss_targets)} ({default_left / max(len(miss_targets), 1):.0%}) "
      f"splits route missing values to the 'yes' branch.")


In [ ]:
# Plot the first booster tree.
booster = native.model_.get_booster()
fig, ax = plt.subplots(figsize=(14, 7))
xgb.plot_tree(booster, num_trees=0, ax=ax)
ax.set_title("XGBoost — first booster tree (default directions visible as 'missing=N' edges)")
plt.tight_layout(); plt.show()


## 8. Feature importance and gain attribution

XGBoost reports per-feature importance based on the total gain contributed across all splits. Compare gain-based vs. cover-based importance to see how second-order weighting changes the ranking.


In [ ]:
importance_gain  = native.model_.get_booster().get_score(importance_type="gain")
importance_cover = native.model_.get_booster().get_score(importance_type="cover")

feats = sorted(set(importance_gain) | set(importance_cover))
import pandas as pd
df = pd.DataFrame({
    "feature": feats,
    "gain":   [importance_gain.get(f, 0.0)  for f in feats],
    "cover":  [importance_cover.get(f, 0.0) for f in feats],
}).sort_values("gain", ascending=False).head(10).reset_index(drop=True)
df


## 9. Exercises

1. **MNAR missingness.** Re-run Section 6 but inject missingness *correlated* with the label (e.g., feature 0 is missing whenever $y = 1$). Does the learned default direction track the label-correlation? How does AUC compare to MCAR?
2. **Approximate vs. exact split finder.** XGBoost's `tree_method="hist"` uses the quantile sketch; `tree_method="exact"` does not. Compare them on a 50k-row dataset for both training time and test AUC. At what `eps` does the sketch lose meaningful accuracy?
3. **Implement your own sketch.** Replace `WeightedQuantileSketch` with a histogram-based variant that uses fixed-width bins instead of weighted quantiles. On what feature distributions does it underperform?

## Takeaways

- **Sparsity-aware split finding** is *learned* imputation; the model picks the best direction at every node.
- **The weighted quantile sketch** exists because the second-order objective is a weighted MSE — without $h_i$, candidate splits would be placed in low-leverage regions.
- The **block-parallel column layout** is what turns a $O(nd \log n)$ algorithm into one that runs in seconds on millions of rows.
- Native missing-value handling is **strictly better** than row-dropping at non-trivial missingness rates.

> **Next week:** LightGBM breaks XGBoost's $O(n \cdot d)$ histogram ceiling with GOSS and EFB.
